# 01 - EDA Flickr8k per CLIP retrieval

Obiettivo: verificare schema, split e subset del dataset `jxie/flickr8k` prima di costruire l'indice text-to-image dell'Esercizio 3.3.

## Perche' serve questa EDA

Prima dell'indicizzazione è utile controllare quali colonne contengono immagini e caption, quale split viene usato e quante immagini entrano nel subset riproducibile.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from datasets import get_dataset_split_names, load_dataset
from omegaconf import OmegaConf

# Risale le cartelle padre finché non trova DLA_LAB2, indipendentemente da dove Jupyter è stato avviato.
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "DLA_LAB2").exists())
lab2_dir = project_root / "DLA_LAB2"
exercise_dir = lab2_dir / "exercise_3_3_clip_retrieval"
if str(lab2_dir) not in sys.path:
    sys.path.insert(0, str(lab2_dir))

from src.clip_retrieval import (
    build_image_records,
    detect_caption_column,
    detect_id_column,
    detect_image_column,
    resolve_split,
)

cfg = OmegaConf.load(exercise_dir / "config.yaml")
cfg

{'dataset': {'name': 'jxie/flickr8k', 'split': 'train', 'max_images': 1000, 'seed': 42}, 'model': {'name': 'openai/clip-vit-base-patch32', 'image_batch_size': 32}, 'retrieval': {'top_k': 10}, 'output': {'dir': 'outputs/clip_retrieval', 'cache_file': 'clip_index.pt'}, 'app': {'server_name': '127.0.0.1', 'server_port': 7860}}

## Schema e split disponibili

In [2]:
dataset_name = str(cfg.dataset.name)
requested_split = str(cfg.dataset.split)
selected_split = resolve_split(dataset_name, requested_split=requested_split)
split_names = get_dataset_split_names(dataset_name)

full_dataset = load_dataset(dataset_name, split=selected_split)
image_column = detect_image_column(full_dataset.features)
caption_column = detect_caption_column(full_dataset.features)
id_column = detect_id_column(full_dataset.features)

schema_df = pd.DataFrame(
    {
        "campo": ["dataset", "split disponibili", "split scelto", "righe split", "colonna immagine", "colonna caption", "colonna id"],
        "valore": [
            dataset_name,
            ", ".join(split_names),
            selected_split,
            len(full_dataset),
            image_column,
            caption_column,
            id_column or "nessuna",
        ],
    }
)
schema_df

,campo,valore
0,dataset,jxie/flickr8k
1,split disponibili,"train, validation, test"
2,split scelto,train
3,righe split,6000
4,colonna immagine,image
5,colonna caption,caption_0
6,colonna id,nessuna


## Subset indicizzato

Il subset viene campionato con seed fisso. In questo modo l'app, il notebook e la cache degli embedding lavorano sulle stesse immagini.

In [3]:
max_images = int(cfg.dataset.max_images)
seed = int(cfg.dataset.seed)
subset_size = min(max_images, len(full_dataset))
subset = full_dataset.shuffle(seed=seed).select(range(subset_size))
records = build_image_records(dataset=subset, split=selected_split)

caption_lengths = pd.Series([len(record.caption.split()) for record in records], name="caption_words")
summary_df = caption_lengths.describe().to_frame().T
summary_df["num_images"] = len(records)
summary_df["seed"] = seed
summary_df

Preparing image records:   0%|          | 0/1000 [00:00<?, ?it/s]

,count,mean,std,min,25%,50%,75%,max,num_images,seed
caption_words,1000.0,12.316,3.942041,4.0,10.0,12.0,14.0,32.0,1000,42


## Esempi di record

In [4]:
examples_df = pd.DataFrame(
    [
        {"item_id": record.item_id, "split": record.split, "caption": record.caption}
        for record in records[:8]
    ]
)
examples_df

,item_id,split,caption
0,train-0,train,Boys with their backs against an incoming wave .
1,train-1,train,A boy in dark blue clothes is kneeling while h...
2,train-2,train,A black and brown dog is running out of the su...
3,train-3,train,A beautiful sunset with three people in a boat...
4,train-4,train,Two men raise their arms atop a snowy mountain .
5,train-5,train,A man dressed like a rockstar poses in front o...
6,train-6,train,one brown and white dog chasing a black and wh...
7,train-7,train,A child is leaping into the air from a sand du...


## Osservazioni

Il dataset `jxie/flickr8k` espone gli split `train`, `validation` e `test`; per l'applicazione e' stato scelto lo split `train`, che contiene 6000 righe. La colonna immagine rilevata e' `image`, mentre la prima caption testuale disponibile e' `caption_0`.

Il subset indicizzato usa 1000 immagini campionate con seed 42. Le caption hanno in media 12.316 parole, con minimo 4 e massimo 32 parole. Questa EDA e' quindi sufficiente come controllo iniziale.